# Theme extraction

## 0. A (not so) small introduction to theme extraction

### 0.1 Abstract
To extract themes from both the contest images (via their textual descriptions) and the user-submitted captions in a scalable and linguistically meaningful way, we rely on **sentence-level embeddings** rather than traditional bag-of-words topic models. 

Note that we do not differenciate captions and images descriptions. They will both inherit the same themes extractions, such that we will be able to compare them together.

### 0.2 Pipeline
Our pipeline is presented like so:

1) **Vector Representation embeddings (Sentence-BERT)**
2) **Unsupervised clustering (Kmeans)**
3) **Theme assignment (TF-IDF based labels)**

Here is a step by step explanation of how it all works:

1) Each description or caption is converted into a dense vector representation using a **Sentence-BERT** model, which captures contextual meaning, humor, and implicit relationships far better than word-frequency-based methods. Our approach is particularly well suited for caption data, which is typically short, creative, and semantically rich.
<br><br>We initially considered classical topic modeling approaches like **Latent Dirichlet Allocation (LDA)**, but LDA’s bag-of-words assumptions and poor performance on short, diverse, humor-oriented texts make it unsuitable for this dataset. We also evaluated **BERTopic**, which combines embeddings with topic modeling, but its reliance on UMAP makes it difficult to scale efficiently to the size of our dataset.

2) Once all texts have been embedded into a shared semantic space, we apply **unsupervised clustering** methods (MiniBatchKMeans) to group semantically similar texts into coherent themes that reflect recurring patterns across contests. By clustering all descriptions and captions globally, rather than per contest for example, we create a unified thematic space that remains consistent throughout the dataset - essential for cross-contest comparisons and further analyses.

3) The last step is to assign these embeddings a meaningful and understandable label. This is where **TF-IDF based labeling** comes in. It will give us a list of the most relevant words from our embeddings. We will then choose the most appropriate one by scoring candidate terms based on their cosine similarity to a theme’s embedding centroid, weighted by how distinctive the term is within the theme compared to the full datatset. The term with the heighest score is then chosen as the label.

This embedding-based clustering pipeline enables us to assign each caption and each contest image to a specific **theme_id**, producing a structured and interpretable representation of the humor of the captions and visual concepts present in the dataset. These theme assignments provide the foundation for following analyses such as exploring funniness-theme correlations, measuring image–caption alignment, and visualizing global humor dynamics.

In [1]:
import json
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

In [2]:
# General paths
ROOT = Path("../../data/newyorker_caption_contest")
CONTESTS_JSON = ROOT / "contests.json"
CSV_DIR = ROOT / "cleaned_data"

# Ooutput directory, create a new output directory if needed
output_dir_base = "themes_extraction_csv"
base_path = ROOT / output_dir_base

idx = None
while True:
    candidate = base_path if idx is None else ROOT / f"{output_dir_base}_{idx}"
    if not candidate.exists():
        # if not present => create folder
        OUTPUT_DIR = candidate
        break
    if not any(candidate.iterdir()):  # if exists and empty => reuse it
        OUTPUT_DIR = candidate
        break
    # exists and non-empty => try next suffix
    idx = 1 if idx is None else idx + 1
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Sentence-BERT model
BERT_MODEL = "sentence-transformers/all-mpnet-base-v2"

# print paths
print(f"Root: {ROOT.resolve()}")
print(f"Contests JSON: {CONTESTS_JSON.resolve()}")
print(f"CSV data folder: {CSV_DIR.resolve()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")  
print(f"\nBERT model: {BERT_MODEL}")

Root: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest
Contests JSON: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/contests.json
CSV data folder: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/cleaned_data
Output folder: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/themes_extraction_csv_2

BERT model: sentence-transformers/all-mpnet-base-v2


## 1. Loading the data

We start by loading our data in a unified way in a custom pandas dataframe. We implemented 2 ways to import our dataset: either from the folder containig all of our data csv and json contest informations or from an output file already generated/formated by this notebook's theme extraction. 

In [3]:
def load_contests(contests_json_path: Path):
    with open(contests_json_path, "r", encoding="utf-8") as f:
        contests = json.load(f)
    return contests

def build_docs_dataframe(debug=False):
    contests = load_contests(CONTESTS_JSON)
    docs = []
    error_csv = [] # to deal with badly formatted csvs

    for contest in tqdm(contests, desc="Step 1: data loading/building"):
        cid = contest["contest_id"]
        raw_meta = contest.get("metadata")
        meta = raw_meta or {}

        # rassembler "image document" par contest
        parts = []
        for key in ["image_descriptions", "image_uncanny_descriptions",
                    "image_locations", "questions"]:
            values = meta.get(key) or []
            parts.extend(values)

        entities = meta.get("entities") or []
        for url in entities:
            last = str(url).rstrip("/").split("/")[-1]  # get last part of URL
            parts.append(last.replace("_", " "))        #erase understcores

        image_text = " ".join(map(str, parts)).strip()  # concat all parts

        # add if non empty (a lot of contests have no image metadata near the end of the dataset)
        if image_text:
            docs.append(
                {
                    "doc_id": f"image_{cid}",
                    "contest_id": cid,
                    "kind": "image",
                    "text": image_text,
                }
            )

        # add caption documents (still per contest)
        csv_path = CSV_DIR / f"{cid}.csv"
        if not csv_path.exists(): # in case using the cleaned csv files
            csv_path = CSV_DIR / f"{cid}_cleaned.csv"
            if not csv_path.exists():
                print(f"Warning: CSV for contest {cid} not found at {CSV_DIR / f'{cid}.csv'} and neither at {csv_path}.")
                continue

        try:
            captions_df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"Error reading CSV for contest {cid} at {csv_path}: {e}")
            error_csv.append(cid)
            continue

        # optional inner tqdm if you want per-caption progress too
        for idx, row in captions_df.iterrows():
            cap_text = str(row.get("caption", "")).strip()
            if not cap_text:
                continue

            docs.append(
                {
                    "doc_id": f"caption_{cid}_{idx}",
                    "contest_id": cid,
                    "kind": "caption",
                    "text": cap_text,
                    "rank": row.get("rank", None),
                    "mean": row.get("mean", None),
                    "precision": row.get("precision", None),
                    "votes": row.get("votes", None),
                    "not_funny": row.get("not_funny", None),
                    "somewhat_funny": row.get("somewhat_funny", None),
                    "funny": row.get("funny", None),
                }
            )

    df = pd.DataFrame(docs)
    
    # keep only expected first 1000 rows for debugging
    if debug:
        df = df.head(5000)
    
    return df, error_csv


# other loading function to load from an output csv that comes from this notebook
def load_docs_with_clean_text(docs_csv_path, debug=False):
    """
    Load the prebuilt docs_with_clean_text.csv file so you don’t have to rebuild
    docs from individual contest files. Returns the dataframe plus an empty
    error list for API parity with build_docs_dataframe.
    """
    docs_csv_path = Path(docs_csv_path)
    if not docs_csv_path.exists():
        raise FileNotFoundError(f"docs_with_clean_text.csv not found at {docs_csv_path}")

    cols = [
        "doc_id", "contest_id", "kind", "text",
        "rank", "mean", "precision", "votes",
        "not_funny", "somewhat_funny", "funny",
        "clean_text",
    ]
    df = pd.read_csv(docs_csv_path)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"docs_with_clean_text.csv is missing expected columns: {missing}")
    
    # remove rows with NAN (the embedding step cannot handle those)
    # df["text"] = df["text"].fillna("")
    
    # keep only expected first 1000 rows for debugging
    if debug:
        df = df.head(5000)

    return df, []

In [4]:
# To load from the original contest files put False
#   => uses CSV_DIR which contains all csv files
# To load from a the prebuilt CSV (that comes from this notebook) put True
#   => uses a docs_with_themes.csv formated file (below step 5.3 and 6)
load_from_docs_csv = True 
use_clean_text = True       # use replace the text column by the clean one

debug = False   # only consider first 1000 rows if True
                # if False, process all rows (~ 2.3 million rows)


if load_from_docs_csv:
    original_docs_df, error_csv = load_docs_with_clean_text("../../data/newyorker_caption_contest/spacy/docs_clean_text.csv", debug=debug)
    
    docs_df = original_docs_df.copy()
    
    if use_clean_text:
        # replace "text" column with "clean_text" column
        docs_df["text"] = docs_df["clean_text"]
        # remove "clean_text" column
        docs_df = docs_df.drop(columns=["clean_text"])
        # remove rows with NAN (the embedding step cannot handle those)
        docs_df["text"] = docs_df["text"].fillna("")
        
else:
    original_docs_df, error_csv = build_docs_dataframe(debug=debug)
    docs_df = original_docs_df.copy()

print("Total documents:", len(docs_df))
print(docs_df["kind"].value_counts())

if error_csv:
    print("Contests with CSV read errors:", error_csv)

Total documents: 2263328
kind
caption    2263088
image          240
Name: count, dtype: int64


In [5]:
original_docs_df.head()

# check that all rows have exactly the same amount of columns/fields
expected_columns = [
    "doc_id", "contest_id", "kind", "text",
    "rank", "mean", "precision", "votes",
    "not_funny", "somewhat_funny", "funny",
]
missing = [c for c in expected_columns if c not in original_docs_df.columns]
if missing:
    raise ValueError(f"original_docs_df is missing expected columns: {missing}")

In [6]:
if use_clean_text:
    docs_df.head()

## 2. Sentence-BERT encoding

This step will take a lot of time to process, on a good laptop it takes around 24 minutes.

So go grab a coffee, open up a good book, or a Selling Sunset episode ... it'll definitely speed up the process. Or invent time travelling, maybe that would work better... maybe... who knows...

In [7]:
model = SentenceTransformer(BERT_MODEL)

def compute_embeddings(texts, batch_size=64):
    # returns a numpy array of shape (N, D)
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # helpful for cosine similarity
    )
    return embeddings

embeddings = compute_embeddings(docs_df["text"].tolist())
print("Embeddings shape:", embeddings.shape)


Batches:   0%|          | 0/35365 [00:00<?, ?it/s]

Embeddings shape: (2263328, 768)


## 3. Global clustering (across all captions + images)

We’ll do global clustering across all documents (captions + images), so they share the same theme space.

For scalability, we will use MiniBatchKMeans. After some fine tuning, we chose a number of clusters of ```n_clusters=80```.

In [8]:
N_CLUSTERS = 80

# apply clustering
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=2048,
    max_iter=100,
    random_state=24,
)
kmeans.fit(embeddings)

theme_labels = kmeans.predict(embeddings)

# save results to dtaframe
docs_df["theme_id"] = theme_labels
docs_df.head()

# copy for good practive
docs_with_themes = docs_df.copy()


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 4. Theme definitions: TF–IDF-based labels per theme_id

Now that we have groups of themes, we want to find what describeds each of them accordingly by finding keywords to described each one of them.

Idea:
- Fit a single TF–IDF over all texts.
- For each theme (cluster), compute the average TF–IDF vector over its documents.
- Take the top N terms → that’s your label candidate.

### 4.1. Fit TF–IDF on all docs

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# fine-tune 
MAX_FEATURES = 100000   # influence the size of the vocabulary => larger = more precise but slower
MIN_DF = 5              # ignore words that appear in <5 documents
NGRAM_RANGE = (1, 3)    # unigrams, bigrams and trigrams, example: "new york city"

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(docs_with_themes["text"])
feature_names = np.array(vectorizer.get_feature_names_out())

print(tfidf_matrix.shape)  # (n_docs, n_features)


(2263328, 100000)


### 4.2. Get top TF–IDF terms per cluster

We’ll compute, for each theme_id:
- the mean TF–IDF vector across its docs
- the top-k features with highest mean score (we'll take top-k = 10)

In [10]:
from collections import defaultdict

def get_top_terms_per_theme(docs_df, tfidf_matrix, feature_names, top_k):
    """
    For each theme in docs_df, compute the top_k terms with highest mean TF-IDF    
    """
    
    theme_to_terms = {}
    theme_ids = sorted(docs_df["theme_id"].unique())

    for theme_id in theme_ids:
        mask = (docs_df["theme_id"] == theme_id).values
        if mask.sum() == 0:
            continue

        # slice TF-IDF rows for this theme
        theme_tfidf = tfidf_matrix[mask]

        # compute mean tf-idf across docs in this theme (1 x n_features)
        mean_tfidf = theme_tfidf.mean(axis=0)  # still a sparse matrix
        mean_tfidf = np.asarray(mean_tfidf).ravel()

        # get top_k indices
        if np.all(mean_tfidf == 0):
            # cluster with weird texts, skip
            theme_to_terms[theme_id] = []
            continue

        top_idx = mean_tfidf.argsort()[::-1][:top_k]
        top_terms = feature_names[top_idx].tolist()
        theme_to_terms[theme_id] = top_terms

    return theme_to_terms

In [11]:
# Run the top terms extraction
theme_to_terms = get_top_terms_per_theme(
    docs_with_themes, tfidf_matrix, feature_names, top_k=10
)

# display first 7 themes with 5 examples of a doc text containing the theme id
for id in list(theme_to_terms.keys()):
    print(f"Theme id {id}:   {theme_to_terms[id]}\nPrompt examples:")
    # print 5 examples of captions per themes:
    i = 0
    for idx, row in docs_with_themes[docs_with_themes["theme_id"] == id].iterrows():
        print(f"  - '{row['text']}'")
        i += 1
        if i >= 5:
            print("")
            break

Theme id 0:   ['dog', 'pet', 'bark', 'leash', 'puppy', 'animal', 'vet', 'know', 'collar', 'think']
Prompt examples:
  - 'bravo pass apathy test fly color dog look concern'
  - 'dog care'
  - 'doggie take notice'
  - 'big white fluffy look angry commuter'
  - 'thank move dog'

Theme id 1:   ['sleep', 'night', 'tired', 'dream', 'bed', 'wake', 'nap', 'asleep', 'morning', 'fall asleep']
Prompt examples:
  - 'want wake city sleep'
  - 'city let sleep'
  - 'sleep number'
  - 'come guy join call flash nap'
  - 'people fall asleep'

Theme id 2:   ['year', 'month', 'season', 'fall', 'date', 'time', 'age', 'think', 'time year', 'spring']
Prompt examples:
  - 'spring'
  - 'year step people time'
  - 'weird year step crack'
  - 'believe view month'
  - 'fall damn'

Theme id 3:   ['work', 'job', 'work home', 'home', 'office', 'say', 'think', 'day', 'bring work', 'worker']
Prompt examples:
  - 'job stepping stone'
  - 'fall work'
  - 'believe pay massage therapist thing'
  - 'think enjoy live work'


In [12]:
# Display the first 10 rows with their theme ids and top terms and text
for i in range(10):
    doc = docs_with_themes.iloc[i]
    theme_id = doc["theme_id"]
    top_terms = theme_to_terms.get(theme_id, [])
    print(f"{doc['doc_id']}: '{doc['text']}'\n => Theme ID {theme_id}: {top_terms}\n")


image_510: 'man relax city street go business step pass man lay middle sidewalk street lay bystander effect crowd'
 => Theme ID 62: ['train', 'walk', 'ride', 'travel', 'way', 'road', 'trip', 'commute', 'flight', 'subway']

caption_510_0: 'congressman obstruction job'
 => Theme ID 64: ['trump', 'president', 'vote', 'election', 'donald', 'republican', 'think', 'white house', 'news', 'say']

caption_510_1: 'mean middle class step'
 => Theme ID 53: ['buy', 'pay', 'price', 'money', 'sale', 'sell', 'stock', 'market', 'think', 'worth']

caption_510_2: 'suit look flat'
 => Theme ID 76: ['wear', 'dress', 'suit', 'pant', 'tie', 'clothe', 'fashion', 'clothing', 'outfit', 'think']

caption_510_3: 'right woman come know'
 => Theme ID 45: ['wife', 'date', 'love', 'relationship', 'husband', 'divorce', 'marriage', 'marry', 'say', 'want']

caption_510_4: 'lie gutter quit drinking'
 => Theme ID 79: ['drink', 'wine', 'coffee', 'beer', 'lemonade', 'bottle', 'glass', 'cup', 'latte', 'water']

caption_510_5

In [13]:
# list all themes with their top terms
for theme_id, top_terms in theme_to_terms.items():
    print(f"Theme ID {theme_id}: {top_terms}")
    
# count number of themes without any top terms
no_terms_count = sum(1 for terms in theme_to_terms.values() if not terms)
print(f"\nNumber of themes without any top terms: {no_terms_count}")

Theme ID 0: ['dog', 'pet', 'bark', 'leash', 'puppy', 'animal', 'vet', 'know', 'collar', 'think']
Theme ID 1: ['sleep', 'night', 'tired', 'dream', 'bed', 'wake', 'nap', 'asleep', 'morning', 'fall asleep']
Theme ID 2: ['year', 'month', 'season', 'fall', 'date', 'time', 'age', 'think', 'time year', 'spring']
Theme ID 3: ['work', 'job', 'work home', 'home', 'office', 'say', 'think', 'day', 'bring work', 'worker']
Theme ID 4: ['clown', 'circus', 'laugh', 'funny', 'cop', 'joke', 'tell', 'party', 'cartoon', 'think']
Theme ID 5: ['room', 'elephant', 'elephant room', 'door', 'talk', 'space', 'talk elephant', 'talk elephant room', 'address', 'think']
Theme ID 6: ['vegan', 'gluten', 'gluten free', 'free', 'vegetarian', 'paleo', 'organic', 'gmo', 'diet', 'lactose']
Theme ID 7: ['think', 'way', 'need', 'know', 'say', 'hell', 'look', 'thing', 'believe', 'tell']
Theme ID 8: ['good', 'way', 'right', 'bad', 'thank', 'thing', 'guy', 'nice', 'boy', 'man']
Theme ID 9: ['bird', 'fly', 'wing', 'pigeon', 'du

Now let's pick one concise label per theme by blending TF‑IDF candidates with embedding similarity and a distinctiveness penalty.

1) cleans candidates
2) for each cluster computes its embedding centroid, embeds each candidate term
3) score each candidates by cosine similarity to the centroid (semantic fit) multiplied by how concentrated the term is in that cluster versus the whole corpus (penalizing generic words).

The highest-scoring term becomes the label, a label_debug DataFrame lets you audit all candidates and scores.

In [14]:
feature_to_idx = {term: i for i, term in enumerate(feature_names)}
global_dfreq = np.asarray((tfidf_matrix > 0).sum(axis=0)).ravel()  # doc freq per term

def rank_theme_labels(
    theme_to_terms,
    docs_df,
    tfidf_matrix,
    embeddings,
    model,
    feature_to_idx,
    global_dfreq,
    min_chars=3,
    pos_filter=False,  # set True to use spaCy en_core_web_sm for POS filtering
):
    """
    Description:
        For each theme, rank candidate terms using a combined score of semantic alignment
        with the cluster centroid and distinctiveness based on local vs global document frequency.
    
    Params:
        theme_to_terms: dict mapping theme_id to list of candidate terms
        docs_df: dataframe with documents and their theme_ids
        tfidf_matrix: sparse matrix of TF-IDF values for all documents
        embeddings: numpy array of shape (n_docs, embedding_dim) with document embeddings
        model: SentenceTransformer model for encoding terms
        feature_to_idx: dict mapping term to its index in the TF-IDF matrix
        global_dfreq: numpy array of shape (n_features,) with global document frequency per term
        min_chars: minimum number of characters for a term to be considered
        pos_filter: whether to filter terms by part-of-speech (requires spaCy)
        
    Returns:
        winners: dict mapping theme_id to the best label and its score
        label_debug: dataframe with all candidate terms and their scores for inspection
    
    """
    nlp = None
    if pos_filter:
        try:
            import spacy
            nlp = spacy.load("en_core_web_sm")
        except Exception as e:
            print("spaCy not available, skipping POS filter:", e)
            pos_filter = False

    def keep_term(term):
        if len(term) < min_chars:
            return False
        if not pos_filter:
            return True
        doc = nlp(term)
        return any(tok.pos_ in {"NOUN", "PROPN", "ADJ"} for tok in doc)

    rows = []
    winners = {}

    for theme_id, raw_candidates in sorted(theme_to_terms.items()):
        # clean + dedupe + ensure the term is in the vocabulary
        candidates = []
        seen = set()
        for t in raw_candidates:
            t = t.strip()
            if t in seen or t not in feature_to_idx:
                continue
            if keep_term(t):
                candidates.append(t)
                seen.add(t)

        mask = (docs_df["theme_id"] == theme_id).values
        if not mask.any() or not candidates:
            continue

        # cluster centroid in embedding space
        centroid = embeddings[mask].mean(axis=0)
        centroid /= np.linalg.norm(centroid) + 1e-12

        # candidate embeddings
        cand_embs = model.encode(candidates, convert_to_numpy=True, normalize_embeddings=True)

        # cluster-specific doc freq
        cluster_dfreq = np.asarray((tfidf_matrix[mask] > 0).sum(axis=0)).ravel()

        best_term, best_score = None, -np.inf
        for term, term_emb in zip(candidates, cand_embs):
            idx = feature_to_idx[term]
            local_freq = cluster_dfreq[idx]
            global_freq = global_dfreq[idx]
            distinctiveness = local_freq / (global_freq + 1e-6)  # step 4: penalize generic words
            sim = float(np.dot(term_emb, centroid))             # step 3: semantic alignment
            score = sim * distinctiveness                       # combined score

            rows.append(
                {
                    "theme_id": theme_id,
                    "term": term,
                    "sim": sim,
                    "local_freq": local_freq,
                    "global_freq": global_freq,
                    "distinctiveness": distinctiveness,
                    "score": score,
                }
            )
            if score > best_score:
                best_term, best_score = term, score

        winners[theme_id] = {"label": best_term, "score": best_score}

    label_debug = pd.DataFrame(rows).sort_values(["theme_id", "score"], ascending=[True, False])
    return winners, label_debug

labels_best, label_debug = rank_theme_labels(
    theme_to_terms,
    docs_with_themes,
    tfidf_matrix,
    embeddings,
    model,
    feature_to_idx,
    global_dfreq,
    min_chars=3,
    pos_filter=False,  # flip to True if spaCy is installed
)

print("Chosen labels:")
for tid, info in sorted(labels_best.items()):
    print(f"Theme {tid:02d}: {info['label']} (score={info['score']:.3f})")

# Inspect candidates and their scores if you want to review:
label_debug.head()


Chosen labels:
Theme 00: dog (score=0.609)
Theme 01: fall asleep (score=0.695)
Theme 02: year (score=0.474)
Theme 03: work home (score=0.641)
Theme 04: clown (score=0.567)
Theme 05: elephant room (score=0.780)
Theme 06: vegan (score=0.726)
Theme 07: hell (score=0.007)
Theme 08: boy (score=0.074)
Theme 09: bird (score=0.646)
Theme 10: haircut (score=0.601)
Theme 11: size matter (score=0.646)
Theme 12: irony (score=0.314)
Theme 14: wifi (score=0.505)
Theme 15: leave (score=0.237)
Theme 16: delivery (score=0.394)
Theme 17: throne (score=0.500)
Theme 18: doctor (score=0.441)
Theme 19: climate change (score=0.799)
Theme 20: wear mask (score=0.720)
Theme 21: color (score=0.698)
Theme 22: know feel (score=0.446)
Theme 23: airbnb (score=0.599)
Theme 24: win (score=0.249)
Theme 25: mustard (score=0.667)
Theme 26: asteroid (score=0.501)
Theme 27: worry (score=0.387)
Theme 28: bread (score=0.471)
Theme 29: keto (score=0.691)
Theme 30: stair (score=0.631)
Theme 31: whale (score=0.565)
Theme 32: so

,theme_id,term,sim,local_freq,global_freq,distinctiveness,score
0,0,dog,0.791499,14256,18522,0.769679,0.609201
4,0,puppy,0.721091,748,928,0.806034,0.581225
1,0,pet,0.747876,2941,4706,0.624947,0.467383
2,0,bark,0.658491,1355,2078,0.652069,0.429382
3,0,leash,0.576722,982,1334,0.736132,0.424544


## 5. Saving results to csv

Export the new dataframe with themes and also a "themes.csv" files containing all themes with mean stats from the dataframe.

In [15]:
####################################
# Map chosen labels to docs (fall back to "miscellaneous" if any theme is missing)
id_to_label = {tid: info["label"] for tid, info in labels_best.items()}
docs_with_themes["theme_label"] = (
    docs_with_themes["theme_id"].map(id_to_label).fillna("miscellaneous").str.title()
)
# Export docs_with_themes with theme_label
cols_export = [
    "doc_id", "contest_id", "kind", "text", "clean_text", "rank", "mean", "precision",
    "votes", "not_funny", "somewhat_funny", "funny", "theme_id", "theme_label",
]
# restore original text and clean_text columns
docs_with_themes["text"] = original_docs_df["text"]
docs_with_themes["clean_text"] = original_docs_df["clean_text"]

docs_with_themes.to_csv(OUTPUT_DIR / "docs_with_themes.csv", columns=cols_export, index=False)


####################################
# Build themes.csv file
themes_summary = (
    docs_with_themes
    .groupby("theme_id")
    .agg(
        nb_instances=("doc_id", "count"),
        mu_rank=("rank", "mean"),
        mu_mean=("mean", "mean"),
        mu_precision=("precision", "mean"),
        mu_not_funny=("not_funny", "mean"),
        mu_somewhat_funny=("somewhat_funny", "mean"),
        mu_funny=("funny", "mean"),
    )
    .reset_index()
)
themes_summary["theme_label"] = themes_summary["theme_id"].map(id_to_label).fillna("miscellaneous").str.title()

# add TF-IDF terms column from 'theme_to_terms' for each themes
themes_summary["TF_IDF_terms"] = themes_summary["theme_id"].map(
    lambda tid: ", ".join(theme_to_terms.get(tid, []))
)
themes_summary = themes_summary[
    ["theme_id", "theme_label", "nb_instances", "mu_rank", "mu_mean", "mu_precision",
     "mu_not_funny", "mu_somewhat_funny", "mu_funny", "TF_IDF_terms"]
]
themes_summary.to_csv(OUTPUT_DIR / "themes.csv", index=False)


####################################
# export label_debug for inspection
label_debug.to_csv(OUTPUT_DIR / "TF_IDF_themes_scores.csv", index=False)


# 7. Other

In [16]:
# def compute_term_mu_mean_by_theme(
#     docs_with_themes: pd.DataFrame,
#     tfidf_matrix,
#     feature_names,
#     tfidf_terms_df: pd.DataFrame,
#     min_docs: int = 5
# ):
#     """
#     Compute the mean humor score for each TF-IDF term within each theme.

#     Parameters
#     ----------
#     docs_with_themes : DataFrame
#         Must contain columns: ['theme_id', 'mean']
#         Row order MUST match tfidf_matrix rows.

#     tfidf_matrix : scipy sparse matrix
#         Shape (n_docs, n_terms)

#     feature_names : list or array
#         TF-IDF feature names aligned with tfidf_matrix columns

#     tfidf_terms_df : DataFrame
#         TF_IDF_themes_scores.csv
#         Must contain columns: ['theme_id', 'term']

#     min_docs : int
#         Minimum number of documents required to compute a stable mean

#     Returns
#     -------
#     DataFrame with columns:
#         ['theme_id', 'term', 'n_docs', 'term_mu_mean']
#     """

#     feature_to_idx = {t: i for i, t in enumerate(feature_names)}

#     theme_ids = docs_with_themes["theme_id"].to_numpy()
#     y = docs_with_themes["mean"].to_numpy()

#     rows = []

#     for theme_id, g in tfidf_terms_df.groupby("theme_id"):
#         theme_mask = theme_ids == theme_id

#         for term in g["term"].unique():
#             idx = feature_to_idx.get(term)
#             if idx is None:
#                 continue

#             term_mask = np.asarray(tfidf_matrix[:, idx] > 0).ravel()
#             m = theme_mask & term_mask

#             n = int(m.sum())
#             if n < min_docs:
#                 continue

#             rows.append({
#                 "theme_id": theme_id,
#                 "term": term,
#                 "n_docs": n,
#                 "term_mu_mean": float(y[m].mean())
#             })

#     return pd.DataFrame(rows)